In [ ]:
import sys
sys.path.insert(0, '..')
import requests
import pandas as pd
import time
from tools.wiki_scrapper import get_augment_mod_names

In [ ]:
augment_mods_list = set(get_augment_mod_names().keys())
print(f"{len(augment_mods_list)} augments mods on wiki")
print(augment_mods_list)

In [ ]:
def get_order_request(url):
    time.sleep(1)
    response = requests.get(url)
    response.raise_for_status()
    return response

def get_bond_mod_offers(augment_mods_list: list[str]):
    df_concat = pd.DataFrame()
    for mod_name in augment_mods_list:
        print(f"Collecting orders from {mod_name}")
        url = f"https://api.warframe.market/v2/orders/item/{mod_name}"
        mod_orders = get_order_request(url)
        df_order = pd.DataFrame(mod_orders.json()["data"])
        df_concat = pd.concat([df_concat, df_order])
    df_concat['user'] = df_concat['user'].str['ingameName']
    df_concat['mod_name'] = mod_name
    df_concat = df_concat[['user', 'rank', 'platinum', 'type', 'quantity', 'perTrade', 'createdAt', 'updatedAt']]
    df_concat = df_concat.sort_values('rank', ascending=False)
    return df_concat

def get_bond_mod_timeseries(augment_mods_list:list[str]):
    collect_modes = ["statistics_closed", "statistics_live"]
    time_period = ["90days", "48hours"]
    df_concat = pd.DataFrame()
    for mod_name in augment_mods_list:
        print(f"Collecting timeseries from {mod_name}")
        url = f"https://api.warframe.market/v1/items/{mod_name}/statistics"
        mod_orders = get_order_request(url)
        for mode in collect_modes:
            for period in time_period:
                df_order = pd.DataFrame(mod_orders.json()["payload"][mode][period])
                df_order['period'] = period
                df_order["mode"] = ("live" if mode == "statistics_live" else "closed")
                df_order["mod_name"] = mod_name
                df_concat = pd.concat([df_concat, df_order])
    df_concat = df_concat.sort_values('datetime', ascending=False)
    return df_concat

In [ ]:
orders_df = get_bond_mod_offers(augment_mods_list)
display(orders_df)

In [ ]:
orders_df.to_csv('../data/augment_orders.csv', sep=';')

In [ ]:
timeseries_df = get_bond_mod_timeseries(augment_mods_list)
display(timeseries_df)

In [ ]:
timeseries_df.to_csv('../data/augment_timeseries.csv', sep=';')